# Lekce 10 - AI agenti v produkci

V této lekci se naučíte **produkční vzory** pro AI agenty s využitím Microsoft Agent Framework (Python). Pokrýváme:

- **Pozorovatelnost** — přidání měření času a logování do interakcí agenta
- **Hodnocení** — použití hodnotícího agenta k ohodnocení kvality odpovědí
- **Řízení nákladů** — strategie pro optimalizaci tokenů a výběr modelu

Scénář je **cestovní agent**, který pomáhá uživatelům plánovat cesty, přičemž navrch je přidáno monitorování a hodnocení.


## Nastavení


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Úvahy při nasazení do produkce

Přechod AI agentů z prototypů do produkčního provozu vyžaduje pečlivou pozornost třem pilířům:

1. **Observabilita** — Potřebujete přehled o tom, co agent dělá, jak dlouho mu to trvá a které nástroje volá. Bez trasování a logování je ladění problémů v produkci téměř nemožné.

2. **Hodnocení** — Automatizované kontroly kvality zajišťují, že odpovědi agenta zůstanou v průběhu času přesné, úplné a užitečné. Hodnotící agent může přiřadit odpovědím skóre podle definovaných kritérií.

3. **Řízení nákladů** — Využití tokenů přímo ovlivňuje náklady. Strategie jako optimalizace promptu, výběr modelu a ukládání do mezipaměti pomáhají udržet výdaje pod kontrolou, aniž by to snižovalo kvalitu.


## Vytvoření pozorovatelného agenta

Definujeme nástroje pro cestování a obalíme volání agenta časováním, abychom mohli sledovat latenci. V produkci byste integrovali OpenTelemetry nebo podobný systém trasování.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vzorové postupy hodnocení

Běžným produkčním vzorem je použití druhého agenta jako **hodnotitele**. Hodnotitel ohodnotí odpověď primárního agenta podle předem definovaných kritérií, jako jsou úplnost, přesnost a užitečnost.

To umožňuje:
- Automatické kontroly kvality před tím, než se odpovědi dostanou k uživatelům
- Detekce regresí při změně promptů nebo modelů
- Průběžné sledování výkonu agenta v čase


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie řízení nákladů

Řízení nákladů je pro produkční AI agenty zásadní. Zde jsou klíčové strategie:

| Strategie | Popis |
|---|---|
| **Optimalizace promptů** | Udržujte systémové instrukce stručné. Odstraňte nadbytečný kontext, abyste snížili počet vstupních tokenů. |
| **Volba modelu** | Používejte menší, levnější modely (např. GPT-4o-mini) pro jednoduché úlohy, jako je klasifikace nebo extrakce, a vyhraďte větší modely pro složité uvažování. |
| **Kešování** | Ukládejte do mezipaměti výsledky nástrojů a časté dotazy, aby se předešlo opakovaným voláním API. |
| **Limity tokenů** | Nastavte limity `max_tokens`, aby nedocházelo k neočekávaně dlouhým odpovědím. |
| **Seskupování** | Seskupujte více uživatelských dotazů do jednoho volání API, kde je to možné. |

V praxi funguje dobře vícestupňový přístup: směrujte jednoduché požadavky na rychlý, levný model a eskalujte pouze složité dotazy na výkonnější model.


## Shrnutí

V této lekci jste se naučili, jak:

1. **Přidat pozorovatelnost** do interakcí agentů pomocí měření času a protokolování, čímž se vytvoří základy pro trasování a monitorování.
2. **Automaticky vyhodnocovat odpovědi agentů** pomocí hodnotícího agenta, který ohodnotí úplnost, přesnost a užitečnost.
3. **Řídit náklady** pomocí optimalizace promptů, výběru modelu, cacheování a rozpočtů tokenů.

Tyto produkční vzory pomáhají zajistit, že vaši AI agenti jsou spolehliví, měřitelní a nákladově efektivní ve velkém měřítku.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Prohlášení o vyloučení odpovědnosti:
Tento dokument byl přeložen pomocí služby pro překlad s využitím umělé inteligence [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho původním jazyce by měl být považován za rozhodující zdroj. U kritických informací doporučujeme využít profesionálního lidského překladu. Nezodpovídáme za žádná nedorozumění nebo mylné výklady vzniklé v důsledku použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
